# Step 6.b — Climate × soil suitability under the AMOC scenarios, global (EC-Earth3 grid)

The product of 5.h applied to the four EC-Earth3 runs (CO2 fixed at the 2025 / 2050 level, without and with the
freshwater hosing that collapses the AMOC; delta BIO variables of 1.3.a, SVM of 3.g, climate probability of 4.g):

**combined = climate probability (4.g, per scenario) × soil rating (2.g, the same for every scenario)**, per crop and
GAEZ input level (low, high). The soil does not change between scenarios, so a difference between two scenarios comes
from the climate only; the soil weights it (a climate gain on poor soil counts little).

- **Novel climate** (4.g: MESS < 0, NT2 > 1 or no support vector within the reach of the SVM kernel): the climate
  probability is 0 there, and the cells are reported apart. The hosing effect is given on all land and on the cells
  assessable in both scenarios of the pair (novel in neither), as in 4.g.
- **Main result: the expected area** Σ combined × land area (Mha), no threshold: the continuous product.
- **Binary maps**: the thresholds of 5.h (TSS max and 95 % of the MIRCA2000 area, chosen on ERA5), applied unchanged
  to every scenario, so that the threshold does not depend on the scenario.
- **Today's crop area**: the mean combined score over the MIRCA2000 harvested area of the crop (weights = hectares):
  how the suitability of the land where the crop grows today changes, without counting the land where it does not.
- **Regions**: Europe (35–72° N, 25° W–45° E, the region of the RCM and LISFLOOD branch), the northern land north of
  50° N, 23.5–50° N, the tropics and the southern extratropics. The AMOC collapse cools the northern hemisphere most
  (North Atlantic, Europe) and shifts the tropical rain belt south.

Outputs in `<out_root>/6_climate_soil_amoc/<route>/<variant>_<radiation>/`: `climate_soil_scenarios_<crop>.nc`
(`combined_<level>`, `suitable_<tss|area>_<level>`, `novel`, dimension `scenario`), `areas_scenarios_<set>.csv`,
`amoc_effect_combined_<set>.csv`, `amoc_effect_regions_<set>.csv`, maps.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters: the set of 4.g and 5.h
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")   # as 4.g and 5.h
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
LEVELS = ["low", "high"]           # GAEZ input level of the soil rating (2.g)
CRITERIA = {"tss": "TSS max", "area": "95 % of MIRCA area"}   # thresholds of 5.h (thresholds_<set>.csv)

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
CLIM_DIR = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
CS_DIR = cu.out_dir(CFG, "5_climate_soil", ROUTE, SET)
OUT = cu.out_dir(CFG, "6_climate_soil_amoc", ROUTE, SET)
SCEN = ["ERA5", *RUNS]

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6   # land area of the cells (Mha)
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
THR = pd.read_csv(CS_DIR / f"thresholds_{SET}.csv", index_col=[0, 1, 2])["threshold"]
print(f"set {SET} | climate {CLIM_DIR} | thresholds {CS_DIR} | outputs {OUT}")

## 1. Combined suitability per scenario

In [ ]:
mha = lambda a: float(np.nansum(LAND_MHA * a))
area_rows, effect_rows, region_rows, MAPS = [], [], [], {}
for crop in CROPS:
    with xr.open_dataset(CLIM_DIR / f"suitability_{crop}.nc") as ds:
        missing = [s for s in SCEN if s not in ds.scenario.values]
        assert not missing, f"{crop}: scenarios {missing} not in 4.g"
        clim = ds["suit_proba"].sel(scenario=SCEN).values.astype(float)       # (scenario, lat, lon), 0 where novel
        novel = ds["novel"].sel(scenario=SCEN).values == 1
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    dv, comb, suit = {}, {}, {}
    for lv in LEVELS:
        soil = np.where(LAND, np.nan_to_num(SOIL[lv][crop].values / 100.0, nan=0.0), np.nan)   # no soil rating -> 0 (5.h)
        comb[lv] = np.where(LAND & np.isfinite(clim), clim * soil, np.nan)
        dv[f"combined_{lv}"] = (("scenario", "lat", "lon"), comb[lv].astype("float32"),
                                {"long_name": f"climate x soil suitability, {lv} input", "note": "0 where the climate is novel"})
        for key, crit in CRITERIA.items():
            t = float(THR.loc[(crop, f"combined ({lv})", crit)])
            suit[(lv, key)] = np.nan_to_num(comb[lv]) >= t
            dv[f"suitable_{key}_{lv}"] = (("scenario", "lat", "lon"), np.where(np.isfinite(comb[lv]), suit[(lv, key)], np.nan).astype("float32"),
                                          {"long_name": f"binary climate x soil suitability, {lv} input, threshold {crit} of 5.h", "threshold": t})
    dv["novel"] = (("scenario", "lat", "lon"), novel.astype("int8"), {"long_name": "novel climate (4.g)"})
    xr.Dataset(dv, coords={"scenario": SCEN, "lat": GRID["lat"], "lon": GRID["lon"]},
               attrs={"crop": crop, "set": SET, "route": ROUTE}).to_netcdf(OUT / f"climate_soil_scenarios_{crop}.nc")
    MAPS[crop] = {"comb": comb, "suit": suit, "novel": novel}

    for k, s in enumerate(SCEN):
        r = {"crop": crop, "scenario": s, "novel (Mha)": mha(novel[k]),
             "MIRCA area of the crop in novel cells (%)": 100 * crop_ha[novel[k]].sum() / crop_ha.sum()}
        for lv in LEVELS:
            r[f"expected area, {lv} (Mha)"] = mha(np.nan_to_num(comb[lv][k]))
            r[f"mean score on today's crop area, {lv}"] = float(np.nansum(np.nan_to_num(comb[lv][k]) * crop_ha) / crop_ha.sum())
            for key in CRITERIA:
                r[f"suitable, {lv}, {key} (Mha)"] = mha(suit[(lv, key)][k])
        area_rows.append(r)

    for hos, orig in PAIRS:
        h, o = SCEN.index(hos), SCEN.index(orig)
        both = LAND & ~(novel[h] | novel[o])
        for lv in LEVELS:
            ch, co = np.nan_to_num(comb[lv][h]), np.nan_to_num(comb[lv][o])
            r = {"crop": crop, "pair": f"{hos} - {orig}", "level": lv, "novel in either (Mha)": mha(LAND & ~both)}
            for lab, m in (("all land", LAND), ("assessable in both", both)):
                eo = mha(co * m)
                r[f"expected area change, {lab} (%)"] = 100 * (mha(ch * m) - eo) / eo if eo else np.nan
            so = float(np.sum(co * crop_ha))
            r["today's crop area: mean score change (%)"] = 100 * (float(np.sum(ch * crop_ha)) - so) / so if so else np.nan
            for key in CRITERIA:
                sh, sw = suit[(lv, key)][h] & both, suit[(lv, key)][o] & both
                r[f"suitable change, {key}, assessable in both (%)"] = 100 * (mha(sh) - mha(sw)) / mha(sw) if mha(sw) else np.nan
                r[f"gained, {key} (Mha)"], r[f"lost, {key} (Mha)"] = mha(sh & ~sw), mha(sw & ~sh)
            effect_rows.append(r)
            for reg, rm in REGIONS.items():
                m = rm & both
                eo = mha(co * m)
                region_rows.append({"crop": crop, "pair": f"{hos} - {orig}", "level": lv, "region": reg,
                                    "expected area, original (Mha)": eo,
                                    "expected area change (%)": 100 * (mha(ch * m) - eo) / eo if eo > 0.5 else np.nan,
                                    "expected area change (Mha)": mha(ch * m) - eo})
    print(f"{crop}: done", flush=True)

## 2. Results

(1) Areas per scenario. (2) Effect of the AMOC collapse (hosing − original): expected area (all land and the cells
assessable in both), mean score on today's crop area, binary areas at the thresholds of 5.h with the land gained and
lost. (3) The expected area change by region (cells assessable in both). Consistency check: the climate-only effect is
in `amoc_effect_<set>.csv` of 4.g.

In [ ]:
areas = pd.DataFrame(area_rows).set_index(["crop", "scenario"])
effect = pd.DataFrame(effect_rows).set_index(["crop", "pair", "level"])
regions = pd.DataFrame(region_rows).set_index(["crop", "pair", "level", "region"])
areas.to_csv(OUT / f"areas_scenarios_{SET}.csv"); effect.to_csv(OUT / f"amoc_effect_combined_{SET}.csv")
regions.to_csv(OUT / f"amoc_effect_regions_{SET}.csv")
with pd.option_context("display.max_rows", 300, "display.max_columns", 40, "display.width", 250):
    print("(1) areas per scenario (Mha), mean combined score on today's crop area")
    display(areas.round(2))
    print("(2) effect of the AMOC collapse, climate x soil")
    display(effect.round(2))
    print("(3) expected area change (%) by region, cells assessable in both, high input")
    display(regions.xs("high", level="level").reset_index().pivot_table(index=["crop", "pair"], columns="region", sort=False,
            values="expected area change (%)")[list(REGIONS)].round(1))

## 3. Maps

Per crop, high input: combined suitability on ERA5 and in the two original runs, the difference hosing − original
(grey: novel in either), and the binary change at the 95 % threshold (gained / lost / suitable in both).

In [ ]:
grey = ListedColormap(["#9e9e9e"])
chg = ListedColormap(["#d7191c", "#e6e6e6", "#1a9641", "#a6d96a"])   # lost | not suitable | gained | suitable in both
lv = "high"
for crop in CROPS:
    C, S, N = MAPS[crop]["comb"][lv], MAPS[crop]["suit"][(lv, "area")], MAPS[crop]["novel"]
    fig, axes = plt.subplots(2, 3, figsize=(22, 7))
    for ax, s in zip(axes[0], ["ERA5", PAIRS[0][1], PAIRS[1][1]]):
        k = SCEN.index(s)
        im = ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(N[k], np.nan, C[k]), vmin=0, vmax=1, cmap="YlGn", shading="nearest")
        ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(N[k], 1.0, np.nan), cmap=grey, shading="nearest")
        ax.set_title(f"{crop}: combined ({lv}), {s} (grey: novel)", fontsize=9)
    plt.colorbar(im, ax=axes[0].tolist(), shrink=0.8)
    for ax, (hos, orig) in zip(axes[1][:2], PAIRS):
        h, o = SCEN.index(hos), SCEN.index(orig)
        either = N[h] | N[o]
        im = ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(either, np.nan, C[h] - C[o]), vmin=-0.3, vmax=0.3, cmap="RdBu", shading="nearest")
        ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(either & LAND, 1.0, np.nan), cmap=grey, shading="nearest")
        ax.set_title(f"{hos} - {orig} (grey: novel in either)", fontsize=9)
    plt.colorbar(im, ax=axes[1][:2].tolist(), shrink=0.8)
    hos, orig = PAIRS[1]
    h, o = SCEN.index(hos), SCEN.index(orig)
    c = np.where(S[o] & ~S[h], 0, np.where(S[h] & ~S[o], 2, np.where(S[h] & S[o], 3, 1))).astype(float)
    axes[1][2].pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, c, np.nan), vmin=-0.5, vmax=3.5, cmap=chg, shading="nearest")
    axes[1][2].set_title(f"{hos} vs {orig}, 95 % threshold: red lost, dark green gained, light green both", fontsize=9)
    plt.savefig(OUT / f"climate_soil_amoc_maps_{SET}_{crop}.png", dpi=80, bbox_inches="tight"); plt.show()

## 4. Final binary maps: climate × soil at the threshold of 5.h, ERA5 and the four runs

The final product per crop and input level (low, high): suitable / not suitable / novel climate at the presence-only
threshold of 5.h (the highest score keeping 95 % of the MIRCA2000 harvested area on ERA5, `MAP_CRITERION = "area"`;
`"tss"` = TSS max, a sensitivity), the same threshold for the five climates. Written in `<outputs>/<crop>/`, five
PNG per input level:
1. `<crop>_<level>_1_ERA5.png`: ERA5 1981–2014;
2. `<crop>_<level>_2_2025_hosing_vs_nohosing.png`: 2025 no hosing, 2025 hosing, land lost and gained with the
   hosing (reference: no hosing);
3. `<crop>_<level>_3_2050_hosing_vs_nohosing.png`: the same for 2050;
4. `<crop>_<level>_4_2025_vs_ERA5.png`: ERA5, land lost and gained from ERA5 in 2025 no hosing and in 2025 hosing;
5. `<crop>_<level>_5_2050_vs_ERA5.png`: the same for 2050.

Change maps: lost (red), gained (dark green), suitable in both (light green), not suitable in both (light grey),
novel climate already in the reference (dark grey) and novel climate only in the new climate (purple: where the run
leaves the climates the model knows); the land gained and lost (Mha, novel cells left out) in the title.

In [ ]:
MAP_LEVELS = ["low", "high"]       # GAEZ input level(s) of the maps
MAP_CRITERION = "area"             # "area" = 95 % of the MIRCA2000 area (main map), "tss" = TSS max
from matplotlib.patches import Patch
BIN = (["#e6e6e6", "#1a9641", "#7f7f7f"], ["not suitable", "suitable", "novel climate"])
CHG = ["#e6e6e6", "#a6d96a", "#d7191c", "#1a9641", "#7f7f7f", "#9e9ac8"]   # + novel in the reference | novel only in the run
chg_labels = lambda ref, run: ["not suitable in both", "suitable in both", "lost", "gained",
                               f"novel climate in {ref}", f"novel climate only in {run}"]
NICE = {"Maize_rf": "maize", "Potato_rf": "potato", "Soybean_rf": "soybean", "Rice_rf": "rice",
        "Wheat_rf": "wheat (winter + spring)", "Wheat-winter_rf": "winter wheat", "Wheat-spring_rf": "spring wheat"}


def panel(ax, classes, colours, labels, title):
    ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, classes, np.nan), vmin=-0.5, vmax=len(colours) - 0.5,
                  cmap=ListedColormap(colours), shading="nearest")
    ax.set_ylim(-60, 85); ax.set_title(title, fontsize=10)
    ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(colours, labels)], loc="lower left", fontsize=7, framealpha=0.9)


def binary(k, S, N):
    return np.where(N[k], 2.0, S[k].astype(float))


def change(ref, new, S, N):
    # 0-3: not suitable / suitable in both, lost, gained; 4: novel in the reference; 5: novel only in the new climate
    a, b, nov = S[ref], S[new], N[ref] | N[new]
    c = np.where(a & b, 1.0, np.where(a & ~b, 2.0, np.where(~a & b, 3.0, 0.0)))
    c = np.where(N[ref], 4.0, np.where(N[new], 5.0, c))
    g, l = mha(~nov & ~a & b), mha(~nov & a & ~b)
    return c, g, l


def save(fig, path):
    fig.savefig(path, dpi=110, bbox_inches="tight"); plt.close(fig)


written = []
for crop in CROPS:
    D = OUT / crop; D.mkdir(exist_ok=True)
    for lv in MAP_LEVELS:
        S, N = MAPS[crop]["suit"][(lv, MAP_CRITERION)], MAPS[crop]["novel"]
        t = float(THR.loc[(crop, f"combined ({lv})", CRITERIA[MAP_CRITERION])])
        head = f"{NICE.get(crop, crop)}, {lv} input, climate x soil >= {t:.2f}"
        e = SCEN.index("ERA5")
        fig, ax = plt.subplots(figsize=(11, 5.2))
        panel(ax, binary(e, S, N), *BIN, f"{head}: ERA5 1981-2014 | suitable {mha(S[e]):.0f} Mha")
        save(fig, D / f"{crop}_{lv}_1_ERA5.png"); written.append(D / f"{crop}_{lv}_1_ERA5.png")
        for n, (hos, orig) in enumerate(PAIRS, start=2):
            yr = hos.split("_")[1]
            h, o = SCEN.index(hos), SCEN.index(orig)
            c, g, l = change(o, h, S, N)
            fig, axes = plt.subplots(1, 3, figsize=(30, 5.2))
            panel(axes[0], binary(o, S, N), *BIN, f"{head}: {yr} no hosing | suitable {mha(S[o]):.0f} Mha")
            panel(axes[1], binary(h, S, N), *BIN, f"{head}: {yr} hosing | suitable {mha(S[h]):.0f} Mha")
            panel(axes[2], c, CHG, chg_labels(f"{yr} no hosing", f"{yr} hosing"),
                  f"{yr} hosing vs no hosing: gained {g:.0f} Mha, lost {l:.0f} Mha")
            f = D / f"{crop}_{lv}_{n}_{yr}_hosing_vs_nohosing.png"; save(fig, f); written.append(f)
        for n, (hos, orig) in enumerate(PAIRS, start=4):
            yr = hos.split("_")[1]
            fig, axes = plt.subplots(1, 3, figsize=(30, 5.2))
            panel(axes[0], binary(e, S, N), *BIN, f"{head}: ERA5 1981-2014 | suitable {mha(S[e]):.0f} Mha")
            for ax, (run, lab) in zip(axes[1:], ((orig, "no hosing"), (hos, "hosing"))):
                c, g, l = change(e, SCEN.index(run), S, N)
                panel(ax, c, CHG, chg_labels("ERA5", f"{yr} {lab}"), f"{yr} {lab} vs ERA5: gained {g:.0f} Mha, lost {l:.0f} Mha")
            f = D / f"{crop}_{lv}_{n}_{yr}_vs_ERA5.png"; save(fig, f); written.append(f)
print(f"{len(written)} maps written in {OUT}/<crop>/")
for f in written[:5]:
    print("  ", f.relative_to(OUT))